# EX1 displacement-only differentiable RT-RPINN (pointwise)\n\nThis notebook is organized into modular blocks following the EX2/EX4 style. The code is unchanged; only the notebook structure is separated for readability.\n

## Import Required Libraries\n\nSet up numerical libraries, plotting backend, random seeds, device selection, and shared Voigt-index conventions.\n

In [ ]:
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Voigt reorder between FE and PINN conventions (both are self-inverse)
_FE2PINN = [0, 1, 2, 5, 4, 3]
_PINN2FE = [0, 1, 2, 5, 4, 3]




## FEDataLoader\n\nLoad finite-element frame data, reconstruct the step/time/temperature sequence, and expose helper routines for domain and boundary data.\n

In [ ]:
# ============================================================================
# Data Loader
# ============================================================================

class FEDataLoader:
    """Load all FE CSV frames; build a time-sorted per-frame sequence."""

    LE_COLS = ['LE11', 'LE22', 'LE33', 'LE12', 'LE13', 'LE23']
    S_COLS  = ['S11',  'S22',  'S33',  'S12',  'S13',  'S23']

    def __init__(self, data_dir, step_frame_time_file=None):
        self.data_dir = Path(data_dir)
        self.sft_file = Path(step_frame_time_file) if step_frame_time_file else None
        self.full_data = None
        self.frames    = []          # sorted list of per-frame dicts
        self.step_frame_time_map = {}
        self.steps_info = {
            1: {'duration': 20.0, 'T_start': 343.0, 'T_end': 343.0},
            2: {'duration': 50.0, 'T_start': 343.0, 'T_end': 298.0},
            3: {'duration':  1.0, 'T_start': 298.0, 'T_end': 298.0},
            4: {'duration': 50.0, 'T_start': 298.0, 'T_end': 353.0},
        }
        self.cum_time = {1: 0.0, 2: 20.0, 3: 70.0, 4: 71.0}

    # ── loading ──────────────────────────────────────────────────────────────

    def load_all_data(self):
        if self.sft_file and self.sft_file.exists():
            df = pd.read_csv(self.sft_file, header=None, names=['Step', 'Frame', 'Time'])
            self.step_frame_time_map = {
                (int(r.Step), int(r.Frame)): float(r.Time) for r in df.itertuples(index=False)
            }
            print(f"Loaded {len(self.step_frame_time_map)} step-frame-time mappings")

        all_dfs = []
        for step in range(1, 5):
            step_files = sorted(
                self.data_dir.glob(f'*_Step-{step}_frame*.csv'),
                key=lambda p: int(p.stem.split('frame')[-1])
            )
            for fp in step_files:
                frame_idx = int(fp.stem.split('frame')[-1])
                if (step, frame_idx) in self.step_frame_time_map:
                    t_f = self.step_frame_time_map[(step, frame_idx)]
                else:
                    dur = self.steps_info[step]['duration']
                    n_f = max(len(step_files) - 1, 1)
                    t_f = self.cum_time[step] + (frame_idx / n_f) * dur

                ts   = self.steps_info[step]
                t_in = t_f - self.cum_time[step]
                temp = (ts['T_start'] + (ts['T_end'] - ts['T_start']) *
                        (t_in / ts['duration']) if ts['duration'] > 0 else ts['T_start'])

                df = pd.read_csv(fp)
                df['Time'] = t_f;  df['Temperature'] = temp
                df['Step'] = step; df['Frame'] = frame_idx
                all_dfs.append(df)

        self.full_data = pd.concat(all_dfs, ignore_index=True)
        self._build_frame_sequence()
        print(f"Loaded {len(self.frames)} frames, {len(self.full_data)} total rows")
        return self.full_data

    def _build_frame_sequence(self):
        nc       = next((c for c in ('NodeLabel', 'Node', 'NID')
                         if c in self.full_data.columns), None)
        le_have  = [c for c in self.LE_COLS if c in self.full_data.columns]
        s_have   = [c for c in self.S_COLS  if c in self.full_data.columns]
        has_le   = len(le_have) == 6
        has_s    = len(s_have)  == 6

        for (step, fi), grp in self.full_data.groupby(['Step', 'Frame'], sort=True):
            g = grp.sort_values(nc) if nc else grp
            d = {
                'time':  float(g['Time'].iloc[0]),
                'temp':  float(g['Temperature'].iloc[0]),
                'step':  int(step),
                'frame': int(fi),
                'x':  g['X'].values.astype(np.float32),
                'y':  g['Y'].values.astype(np.float32),
                'z':  g['Z'].values.astype(np.float32),
                'u':  g[['U1', 'U2', 'U3']].values.astype(np.float32),  # (N,3) mm
                # FE log-strain (material frame assumed, FE Voigt order)
                'le': g[le_have].values.astype(np.float32) if has_le else None,
                # FE Cauchy stress converted MPa→Pa (FE Voigt order)
                's':  g[s_have].values.astype(np.float32) * 1e6 if has_s else None,
            }
            if nc:
                d['node_ids'] = g[nc].values.astype(np.int32)
            self.frames.append(d)

        self.frames.sort(key=lambda f: f['time'])

    # ── helpers ──────────────────────────────────────────────────────────────

    def get_domain_bounds(self):
        d = self.full_data
        return {'x_min': float(d['X'].min()), 'x_max': float(d['X'].max()),
                'y_min': float(d['Y'].min()), 'y_max': float(d['Y'].max()),
                'z_min': float(d['Z'].min()), 'z_max': float(d['Z'].max()),
                't_min': float(d['Time'].min()), 't_max': float(d['Time'].max())}

    def get_left_face_data(self):
        x_min = self.full_data['X'].min()
        return self.full_data[np.abs(self.full_data['X'] - x_min) < 0.01]




## Material Parameters\n\nDefine the thermo-viscoelastic SMPC material constants, Prony spectrum, thermal expansion, and WLF/Arrhenius shift-factor functions.\n

In [ ]:
# ============================================================================
# Material Parameters  (identical to original)
# ============================================================================

class MaterialParameters:
    def __init__(self):
        self.T_ref    = 323.0
        self.T_switch = 317.4
        self.C1 = 14.8;  self.C2 = 45.6
        self.E_arrhenius = 27403.3;  self.T_arr_ref = 336.0

        self.rho = np.array([0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])

        self.C11_inf   = 10319.1
        self.C11_prony = np.array([200.254, 89.0021, 288.039, 322.882, 29.7953, 1.08799])
        self.C12_inf   = 1.77138
        self.C12_prony = np.array([183.83, 82.8455, 272.998, 318.639, 30.0832, 1.0973])
        self.C22_inf   = 2.83844
        self.C22_prony = np.array([293.707, 132.426, 436.637, 510.272, 48.2092, 1.75608])
        self.C23_inf   = 1.82402
        self.C23_prony = np.array([188.472, 84.9984, 280.342, 327.818, 30.9813, 1.1279])
        self.C66_inf   = 0.530688
        self.C66_prony = np.array([55.3091, 24.9132, 82.0325, 95.5597, 9.01153, 0.328643])

        Tp1 = 3.7e-6;  Tp2 = 0.00012
        self.Tp = np.array([Tp1, Tp2, Tp2, 0.0, 0.0, 0.0])

        for attr in ('C11_inf','C11_prony','C12_inf','C12_prony','C22_inf','C22_prony',
                     'C23_inf','C23_prony','C66_inf','C66_prony'):
            setattr(self, attr, getattr(self, attr) * 1e6)  # MPa → Pa

        self.N_prony = 6
        self.tau_0   = self.rho
        self.L_ref   = 33.0         # mm
        self.E_ref   = self.C11_inf # Pa

    def shift_factor(self, T):
        if isinstance(T, torch.Tensor):
            T = torch.clamp(T, 250.0, 400.0)
            aT_wlf = 10.0 ** (-self.C1 * (T - self.T_ref) / (self.C2 + T - self.T_ref))
            aT_arr = torch.exp(self.E_arrhenius * (1.0/T - 1.0/self.T_arr_ref))
            return torch.where(T > self.T_switch, aT_wlf, aT_arr)
        else:
            T = np.clip(float(T), 250.0, 400.0)
            if T > self.T_switch:
                return 10.0 ** (-self.C1 * (T - self.T_ref) / (self.C2 + T - self.T_ref))
            return np.exp(self.E_arrhenius * (1.0/T - 1.0/self.T_arr_ref))




## Model Architecture\n\nDefine the displacement-only RT-RPINN backbones, including pointwise, LSTM-based, and three-head LSTM variants.\n

In [ ]:
# ============================================================================


"""Displacement-only reduced-time recursive PINN core for EX1 and EX3.

The neural model returns displacement only. Strain is obtained by automatic
differentiation, the generalized-Maxwell state is advanced with the exact
piecewise-linear update used in the paper, and stress/equilibrium follow from
that recursively propagated state. No FE strain, FE stress, teacher-forced
state, or independently predicted internal variable enters training.
"""


import csv
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class PointwiseDisplacementPINN(nn.Module):
    """Pointwise MLP mapping (x,y,z,t,T) to displacement only."""

    def __init__(self, bounds, hidden, output_scale=1.0):
        super().__init__()
        self.bounds = bounds
        self.output_scale = float(output_scale)
        layers = []
        width_in = 5
        for width in hidden:
            layers.extend((nn.Linear(width_in, width), nn.Tanh()))
            width_in = width
        layers.append(nn.Linear(width_in, 3))
        self.net = nn.Sequential(*layers)
        self._reset_parameters()

    def _reset_parameters(self):
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def _normalize(self, x, y, z, t, temperature):
        b = self.bounds

        def scale(value, lo, hi):
            return 2.0 * (value - lo) / max(float(hi - lo), 1.0e-12) - 1.0

        return torch.stack(
            (
                scale(x, b['x_min'], b['x_max']),
                scale(y, b['y_min'], b['y_max']),
                scale(z, b['z_min'], b['z_max']),
                scale(t, b['t_min'], b['t_max']),
                scale(temperature, 298.0, 353.0),
            ),
            dim=-1,
        )

    def forward_history(self, x, y, z, times, temperatures):
        batch = x.shape[0]
        n_time = times.numel()
        xx = x[:, None].expand(batch, n_time)
        yy = y[:, None].expand(batch, n_time)
        zz = z[:, None].expand(batch, n_time)
        tt = times[None, :].expand(batch, n_time)
        temp = temperatures[None, :].expand(batch, n_time)
        features = self._normalize(xx, yy, zz, tt, temp)
        return self.output_scale * self.net(features.reshape(-1, 5)).reshape(
            batch, n_time, 3
        )


class LSTMDisplacementPINN(nn.Module):
    """Shared recurrent displacement encoder with no constitutive-state head."""

    def __init__(self, bounds, encoder_widths=(256, 256, 256),
                 hidden_size=512, num_layers=3, decoder_widths=(256, 128),
                 output_scale=1.0, dropout=0.02):
        super().__init__()
        self.bounds = bounds
        self.output_scale = float(output_scale)
        self.encoder = self._mlp(5, encoder_widths)
        encoded = encoder_widths[-1]
        self.lstm = nn.LSTM(
            encoded,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.decoder = self._mlp(hidden_size, decoder_widths, output_dim=3)
        self._reset_parameters()

    @staticmethod
    def _mlp(input_dim, widths, output_dim=None):
        layers = []
        previous = input_dim
        for width in widths:
            layers.extend((nn.Linear(previous, width), nn.Tanh()))
            previous = width
        if output_dim is not None:
            layers.append(nn.Linear(previous, output_dim))
        return nn.Sequential(*layers)

    def _reset_parameters(self):
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def _features(self, x, y, z, times, temperatures):
        b = self.bounds

        def scale(value, lo, hi):
            return 2.0 * (value - lo) / max(float(hi - lo), 1.0e-12) - 1.0

        batch = x.shape[0]
        n_time = times.numel()
        return torch.stack(
            (
                scale(x, b['x_min'], b['x_max'])[:, None].expand(batch, n_time),
                scale(y, b['y_min'], b['y_max'])[:, None].expand(batch, n_time),
                scale(z, b['z_min'], b['z_max'])[:, None].expand(batch, n_time),
                scale(times, b['t_min'], b['t_max'])[None, :].expand(batch, n_time),
                scale(temperatures, 298.0, 353.0)[None, :].expand(batch, n_time),
            ),
            dim=-1,
        )

    def forward_history(self, x, y, z, times, temperatures):
        features = self._features(x, y, z, times, temperatures)
        encoded = self.encoder(features)
        recurrent, _ = self.lstm(encoded)
        return self.output_scale * self.decoder(recurrent)


class ThreeHeadLSTMDisplacementPINN(LSTMDisplacementPINN):
    """Independent recurrent displacement heads; none predicts q."""

    def __init__(self, bounds, encoder_widths=(256, 256, 256),
                 hidden_size=512, num_layers=3, decoder_widths=(256, 128),
                 output_scale=1.0, dropout=0.02):
        nn.Module.__init__(self)
        self.bounds = bounds
        self.output_scale = float(output_scale)
        self.encoders = nn.ModuleList()
        self.lstms = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for _ in range(3):
            self.encoders.append(self._mlp(5, encoder_widths))
            self.lstms.append(
                nn.LSTM(
                    encoder_widths[-1],
                    hidden_size,
                    num_layers=num_layers,
                    batch_first=True,
                    dropout=dropout if num_layers > 1 else 0.0,
                )
            )
            self.decoders.append(
                self._mlp(hidden_size, decoder_widths, output_dim=1)
            )
        self._reset_parameters()

    def forward_history(self, x, y, z, times, temperatures):
        features = self._features(x, y, z, times, temperatures)
        outputs = []
        for encoder, lstm, decoder in zip(
            self.encoders, self.lstms, self.decoders
        ):
            recurrent, _ = lstm(encoder(features))
            outputs.append(decoder(recurrent))
        return self.output_scale * torch.cat(outputs, dim=-1)




## Solver and Loss Functions\n\nEvaluate strains by automatic differentiation, propagate reduced-time internal variables, compute stresses/residuals, and assemble training losses.\n

In [ ]:
class DisplacementRecursiveSolver:
    """Differentiable displacement -> strain -> q -> stress -> equilibrium path."""

    def __init__(
        self,
        model,
        material,
        fe_loader,
        bounds,
        device,
        spatial_ratio=1.0,
        temporal_ratio=1.0,
        seed=42,
        component_weights=(1.0, 10.0, 300.0),
        lambda_data=1.0,
        lambda_equilibrium=1.0,
        lambda_bc_left=1.0,
        lambda_bc_right=1.0,
        hole_center=None,
        hole_radius=None,
    ):
        self.model = model.to(device)
        self.material = material
        self.loader = fe_loader
        self.bounds = bounds
        self.device = device
        self.seed = int(seed)
        self.rng = np.random.default_rng(self.seed)
        self.component_weights = torch.tensor(
            component_weights, dtype=torch.float32, device=device
        ).view(1, 1, 3)
        self.lambda_data = float(lambda_data)
        self.lambda_equilibrium = float(lambda_equilibrium)
        self.lambda_bc_left = float(lambda_bc_left)
        self.lambda_bc_right = float(lambda_bc_right)
        self.hole_center = hole_center
        self.hole_radius = hole_radius

        self.frames = list(fe_loader.frames)
        if not self.frames:
            raise ValueError('FEDataLoader.frames is empty; call load_all_data first.')
        self._check_frame_alignment()
        self.frame_indices = self._select_frames(temporal_ratio)
        self.node_indices = self._select_nodes(spatial_ratio)
        self.left_indices, self.right_indices, self.interior_indices = self._faces()

        self.times = torch.tensor(
            [self.frames[i]['time'] for i in self.frame_indices],
            dtype=torch.float32,
            device=device,
        )
        self.temperatures = torch.tensor(
            [self.frames[i]['temp'] for i in self.frame_indices],
            dtype=torch.float32,
            device=device,
        )
        self.u_observed = np.stack(
            [self.frames[i]['u'] for i in self.frame_indices], axis=1
        ).astype(np.float32)
        self.loss_history = []
        self.loss_components_history = []
        self._build_material_tensors()

        print(
            f'Recursive history: {len(self.frame_indices)}/{len(self.frames)} frames; '
            f'observed nodes: {len(self.node_indices)}/{len(self.frames[0]["x"])}'
        )
        print('State source: network displacement history only (no FE strain/stress/q).')

    def _check_frame_alignment(self):
        reference = self.frames[0]
        n_nodes = len(reference['x'])
        reference_ids = reference.get('node_ids')
        for frame in self.frames[1:]:
            if len(frame['x']) != n_nodes:
                raise ValueError('Frames do not contain the same number of nodes.')
            if reference_ids is not None and not np.array_equal(
                reference_ids, frame.get('node_ids')
            ):
                raise ValueError('Node ordering differs among FE frames.')

    def _select_frames(self, ratio):
        n_total = len(self.frames)
        n_keep = min(n_total, max(2, int(round(float(ratio) * n_total))))
        frame_times = np.asarray([frame['time'] for frame in self.frames])
        transition_times = (0.0, 20.0, 70.0, 71.0, 121.0)
        mandatory = {
            int(np.argmin(np.abs(frame_times - transition)))
            for transition in transition_times
            if frame_times.min() <= transition <= frame_times.max()
        }
        candidates = np.unique(np.linspace(0, n_total - 1, n_keep, dtype=int)).tolist()
        selected = sorted(mandatory)
        for candidate in candidates:
            if candidate not in mandatory and len(selected) < n_keep:
                selected.append(candidate)
        if len(selected) < n_keep:
            remaining = np.setdiff1d(np.arange(n_total), np.asarray(selected, dtype=int))
            fill = np.linspace(0, len(remaining) - 1, n_keep - len(selected), dtype=int)
            selected.extend(remaining[fill].tolist())
        return sorted(selected[:n_keep])

    def _select_nodes(self, ratio):
        frame = self.frames[0]
        n_total = len(frame['x'])
        n_keep = min(n_total, max(1, int(round(float(ratio) * n_total))))
        all_indices = np.arange(n_total)
        if self.hole_center is None or self.hole_radius is None:
            return np.sort(self.rng.choice(all_indices, n_keep, replace=False))

        x, y = frame['x'], frame['y']
        cx, cy = self.hole_center
        radius = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
        near_hole = all_indices[
            (radius >= self.hole_radius) & (radius <= self.hole_radius + 2.0)
        ]
        outer = all_indices[
            (np.abs(x - self.bounds['x_min']) < 1.0e-5)
            | (np.abs(x - self.bounds['x_max']) < 1.0e-5)
            | (np.abs(y - self.bounds['y_min']) < 1.0e-5)
            | (np.abs(y - self.bounds['y_max']) < 1.0e-5)
        ]
        selected = []
        for candidates, fraction in ((near_hole, 0.4), (outer, 0.3)):
            available = np.setdiff1d(candidates, np.asarray(selected, dtype=int))
            count = min(len(available), int(round(fraction * n_keep)))
            if count:
                selected.extend(self.rng.choice(available, count, replace=False).tolist())
        remaining = np.setdiff1d(all_indices, np.asarray(selected, dtype=int))
        if len(selected) < n_keep:
            selected.extend(
                self.rng.choice(remaining, n_keep - len(selected), replace=False).tolist()
            )
        return np.sort(np.asarray(selected[:n_keep], dtype=int))

    def _faces(self):
        frame = self.frames[0]
        x = frame['x']
        tol = max(1.0e-5, 1.0e-5 * (self.bounds['x_max'] - self.bounds['x_min']))
        left = np.flatnonzero(np.abs(x - self.bounds['x_min']) <= tol)
        right = np.flatnonzero(np.abs(x - self.bounds['x_max']) <= tol)
        boundary = np.union1d(left, right)
        interior = np.setdiff1d(np.arange(len(x)), boundary)
        return left, right, interior

    def _build_material_tensors(self):
        mat = self.material
        self.rho = torch.tensor(mat.rho, dtype=torch.float32, device=self.device)
        self.alpha = torch.tensor(mat.Tp, dtype=torch.float32, device=self.device)

        def tensor(name):
            return torch.as_tensor(
                getattr(mat, name), dtype=torch.float32, device=self.device
            )

        self.C_inf = {
            '11': tensor('C11_inf'), '12': tensor('C12_inf'),
            '22': tensor('C22_inf'), '23': tensor('C23_inf'),
            '66': tensor('C66_inf'),
        }
        self.C_branch = {
            '11': tensor('C11_prony'), '12': tensor('C12_prony'),
            '22': tensor('C22_prony'), '23': tensor('C23_prony'),
            '66': tensor('C66_prony'),
        }
        angle = np.deg2rad(45.0)
        c, s = float(np.cos(angle)), float(np.sin(angle))
        self.Q = torch.tensor(
            [[c, s, 0.0], [-s, c, 0.0], [0.0, 0.0, 1.0]],
            dtype=torch.float32,
            device=self.device,
        )
        self.L_ref = float(mat.L_ref)
        self.E_ref = float(mat.E_ref)

    def _coordinates(self, indices, requires_grad=True):
        frame = self.frames[0]
        values = []
        for name in ('x', 'y', 'z'):
            tensor = torch.tensor(
                frame[name][indices], dtype=torch.float32, device=self.device
            )
            tensor.requires_grad_(requires_grad)
            values.append(tensor)
        return values

    @staticmethod
    def _gradient(output, coordinate, create_graph=True, retain_graph=True):
        return torch.autograd.grad(
            output,
            coordinate,
            grad_outputs=torch.ones_like(output),
            create_graph=create_graph,
            retain_graph=retain_graph,
        )[0]

    def displacement_to_material_strain(self, displacement, x, y, z):
        strain_steps = []
        for n in range(displacement.shape[1]):
            u1, u2, u3 = displacement[:, n, 0], displacement[:, n, 1], displacement[:, n, 2]
            u1x = self._gradient(u1, x)
            u1y = self._gradient(u1, y)
            u1z = self._gradient(u1, z)
            u2x = self._gradient(u2, x)
            u2y = self._gradient(u2, y)
            u2z = self._gradient(u2, z)
            u3x = self._gradient(u3, x)
            u3y = self._gradient(u3, y)
            u3z = self._gradient(u3, z)
            strain_tensor = torch.zeros(
                len(x), 3, 3, dtype=torch.float32, device=self.device
            )
            strain_tensor[:, 0, 0] = u1x
            strain_tensor[:, 1, 1] = u2y
            strain_tensor[:, 2, 2] = u3z
            strain_tensor[:, 0, 1] = strain_tensor[:, 1, 0] = 0.5 * (u1y + u2x)
            strain_tensor[:, 0, 2] = strain_tensor[:, 2, 0] = 0.5 * (u1z + u3x)
            strain_tensor[:, 1, 2] = strain_tensor[:, 2, 1] = 0.5 * (u2z + u3y)
            material_tensor = self.Q @ strain_tensor @ self.Q.T
            engineering = torch.stack(
                (
                    material_tensor[:, 0, 0], material_tensor[:, 1, 1],
                    material_tensor[:, 2, 2], 2.0 * material_tensor[:, 1, 2],
                    2.0 * material_tensor[:, 0, 2], 2.0 * material_tensor[:, 0, 1],
                ),
                dim=1,
            )
            thermal = self.alpha * (self.temperatures[n] - self.material.T_ref)
            strain_steps.append(engineering - thermal[None, :])
        return torch.stack(strain_steps, dim=1)

    def reduced_time_state(self, strain_history):
        batch, n_time, _, n_branch = (
            strain_history.shape[0], strain_history.shape[1], 6, len(self.rho)
        )
        state = torch.zeros(
            batch, 6, n_branch, dtype=torch.float32, device=self.device
        )
        states = [state]
        for n in range(1, n_time):
            dt = torch.clamp(self.times[n] - self.times[n - 1], min=0.0)
            shift = torch.clamp(
                self.material.shift_factor(self.temperatures[n]),
                min=1.0e-30,
                max=1.0e30,
            )
            reduced_increment = dt / shift
            ratio = torch.clamp(reduced_increment / self.rho, min=0.0, max=80.0)
            gamma = torch.exp(-ratio)
            beta = torch.where(
                ratio < 1.0e-5,
                1.0 - 0.5 * ratio + ratio.square() / 6.0,
                -torch.expm1(-ratio) / ratio,
            )
            delta_strain = strain_history[:, n] - strain_history[:, n - 1]
            state = gamma[None, None, :] * state + beta[None, None, :] * delta_strain[:, :, None]
            states.append(state)
        return torch.stack(states, dim=1)

    def material_stress(self, strain, state):
        ci, cb = self.C_inf, self.C_branch
        e11, e22, e33, g23, g13, g12 = strain.unbind(dim=-1)

        s11 = ci['11'] * e11 + ci['12'] * (e22 + e33)
        s22 = ci['12'] * e11 + ci['22'] * e22 + ci['23'] * e33
        s33 = ci['12'] * e11 + ci['23'] * e22 + ci['22'] * e33
        s23 = 0.5 * (ci['22'] - ci['23']) * g23
        s13 = ci['66'] * g13
        s12 = ci['66'] * g12

        q11, q22, q33, q23, q13, q12 = state.unbind(dim=-2)
        s11 = s11 + torch.sum(cb['11'] * q11 + cb['12'] * (q22 + q33), dim=-1)
        s22 = s22 + torch.sum(cb['12'] * q11 + cb['22'] * q22 + cb['23'] * q33, dim=-1)
        s33 = s33 + torch.sum(cb['12'] * q11 + cb['23'] * q22 + cb['22'] * q33, dim=-1)
        s23 = s23 + torch.sum(0.5 * (cb['22'] - cb['23']) * q23, dim=-1)
        s13 = s13 + torch.sum(cb['66'] * q13, dim=-1)
        s12 = s12 + torch.sum(cb['66'] * q12, dim=-1)
        return torch.stack((s11, s22, s33, s23, s13, s12), dim=-1)

    def material_to_global_stress(self, material_stress):
        tensor = torch.zeros(
            *material_stress.shape[:-1], 3, 3,
            dtype=torch.float32,
            device=self.device,
        )
        tensor[..., 0, 0] = material_stress[..., 0]
        tensor[..., 1, 1] = material_stress[..., 1]
        tensor[..., 2, 2] = material_stress[..., 2]
        tensor[..., 1, 2] = tensor[..., 2, 1] = material_stress[..., 3]
        tensor[..., 0, 2] = tensor[..., 2, 0] = material_stress[..., 4]
        tensor[..., 0, 1] = tensor[..., 1, 0] = material_stress[..., 5]
        global_tensor = self.Q.T @ tensor @ self.Q
        return torch.stack(
            (
                global_tensor[..., 0, 0], global_tensor[..., 1, 1],
                global_tensor[..., 2, 2], global_tensor[..., 1, 2],
                global_tensor[..., 0, 2], global_tensor[..., 0, 1],
            ),
            dim=-1,
        )

    def field_state_history(self, indices):
        x, y, z = self._coordinates(indices, requires_grad=True)
        displacement = self.model.forward_history(
            x, y, z, self.times, self.temperatures
        )
        strain = self.displacement_to_material_strain(displacement, x, y, z)
        state = self.reduced_time_state(strain)
        stress_material = self.material_stress(strain, state)
        stress_global = self.material_to_global_stress(stress_material)
        return x, y, z, displacement, strain, state, stress_global

    def equilibrium_loss(self, stress, x, y, z, n_time_samples):
        candidates = np.arange(1, stress.shape[1])
        count = min(len(candidates), int(n_time_samples))
        chosen = self.rng.choice(candidates, count, replace=False) if count else []
        residuals = []
        for n in np.sort(chosen):
            s11, s22, s33, s23, s13, s12 = stress[:, n].unbind(dim=-1)
            r1 = self._gradient(s11, x) + self._gradient(s12, y) + self._gradient(s13, z)
            r2 = self._gradient(s12, x) + self._gradient(s22, y) + self._gradient(s23, z)
            r3 = self._gradient(s13, x) + self._gradient(s23, y) + self._gradient(s33, z)
            residuals.append(torch.stack((r1, r2, r3), dim=-1))
        if not residuals:
            return torch.zeros((), dtype=torch.float32, device=self.device)
        scale = self.E_ref / self.L_ref
        return torch.mean((torch.stack(residuals, dim=1) / scale) ** 2)

    def boundary_loss(self, indices, component=None, max_nodes=128):
        if len(indices) == 0:
            return torch.zeros((), dtype=torch.float32, device=self.device)
        selected = indices
        if len(selected) > max_nodes:
            selected = self.rng.choice(selected, max_nodes, replace=False)
        x, y, z = self._coordinates(selected, requires_grad=False)
        prediction = self.model.forward_history(x, y, z, self.times, self.temperatures)
        observed = torch.tensor(
            self.u_observed[selected], dtype=torch.float32, device=self.device
        )
        if component is not None:
            prediction = prediction[..., component:component + 1]
            observed = observed[..., component:component + 1]
        return torch.mean(((prediction - observed) / self.L_ref) ** 2)

    def train(
        self,
        epochs,
        node_batch=128,
        pde_time_samples=6,
        learning_rate=1.0e-4,
        log_interval=100,
        output_dir=None,
    ):
        optimizer = Adam(self.model.parameters(), lr=learning_rate)
        scheduler = CosineAnnealingLR(
            optimizer, T_max=max(int(epochs), 1), eta_min=learning_rate * 0.05
        )
        start = time.time()
        output_dir = Path(output_dir) if output_dir else None
        if output_dir:
            output_dir.mkdir(parents=True, exist_ok=True)

        for epoch in range(int(epochs)):
            self.model.train()
            optimizer.zero_grad(set_to_none=True)
            candidates = np.intersect1d(self.node_indices, self.interior_indices)
            batch = self.rng.choice(
                candidates, min(int(node_batch), len(candidates)), replace=False
            )
            x, y, z, displacement, _, _, stress = self.field_state_history(batch)
            observed = torch.tensor(
                self.u_observed[batch], dtype=torch.float32, device=self.device
            )
            data_loss = torch.mean(
                (((displacement - observed) / self.L_ref) * self.component_weights) ** 2
            )
            equilibrium_loss = self.equilibrium_loss(
                stress, x, y, z, pde_time_samples
            )
            left_loss = self.boundary_loss(self.left_indices, component=None)
            right_loss = self.boundary_loss(self.right_indices, component=0)
            total = (
                self.lambda_data * data_loss
                + self.lambda_equilibrium * equilibrium_loss
                + self.lambda_bc_left * left_loss
                + self.lambda_bc_right * right_loss
            )
            if not torch.isfinite(total):
                raise FloatingPointError(f'Non-finite loss at epoch {epoch}.')
            total.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)
            optimizer.step()
            scheduler.step()

            row = {
                'epoch': epoch,
                'total': float(total.detach()),
                'data': float(data_loss.detach()),
                'equilibrium': float(equilibrium_loss.detach()),
                'bc_left': float(left_loss.detach()),
                'bc_right': float(right_loss.detach()),
                'lr': optimizer.param_groups[0]['lr'],
            }
            self.loss_history.append(row['total'])
            self.loss_components_history.append(row)
            if epoch % int(log_interval) == 0 or epoch == int(epochs) - 1:
                print(
                    f"[Epoch {epoch:6d}] total={row['total']:.4e} "
                    f"data={row['data']:.4e} eq={row['equilibrium']:.4e} "
                    f"bcL={row['bc_left']:.4e} bcR={row['bc_right']:.4e}"
                )

        print(f'Training complete: {time.time() - start:.1f}s')
        if output_dir:
            self.save_loss_history(output_dir / 'training_history.csv')
            torch.save(self.model.state_dict(), output_dir / 'model.pth')

    def save_loss_history(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        if not self.loss_components_history:
            return
        with path.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(
                stream, fieldnames=list(self.loss_components_history[0].keys())
            )
            writer.writeheader()
            writer.writerows(self.loss_components_history)


"""Train displacement-only reduced-time recursive models for EX1."""

import argparse
from pathlib import Path

import numpy as np
import torch





## Training Configuration and Execution\n\nConfigure the benchmark, instantiate the data loader/model/solver, train the network, and save outputs.\n

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', choices=('pointwise', 'lstm', 'threehead'), default='pointwise')
    parser.add_argument('--epochs', type=int, default=10000)
    parser.add_argument('--node-batch', type=int, default=64)
    parser.add_argument('--pde-times', type=int, default=4)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--output-dir', type=str, default=None)
    args = parser.parse_args(args=[])
    set_seed(args.seed)

    script_dir = NOTEBOOK_DIR
    loader = FEDataLoader(script_dir / 'EX-1-RESULTS', script_dir / 'step-frame-time.csv')
    loader.load_all_data()
    bounds = loader.get_domain_bounds()
    material = MaterialParameters()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_scale = max(1.0, 1.1 * max(float(np.max(np.abs(frame['u']))) for frame in loader.frames))

    if args.model == 'pointwise':
        model = PointwiseDisplacementPINN(bounds, hidden=(256,) * 5, output_scale=output_scale)
        component_weights = (1.0, 10.0, 300.0)
        lambda_equilibrium = 0.7
    elif args.model == 'lstm':
        model = LSTMDisplacementPINN(bounds, output_scale=output_scale)
        component_weights = (1.0, 10.0, 100.0)
        lambda_equilibrium = 1.0
    else:
        model = ThreeHeadLSTMDisplacementPINN(bounds, output_scale=output_scale)
        component_weights = (1.0, 10.0, 100.0)
        lambda_equilibrium = 1.0

    # The existing recursive EX1 pilot used 80 ordered frames. Retaining the
    # same history density keeps the new comparison focused on removing q
    # prediction and teacher forcing.
    temporal_ratio = min(1.0, 80.0 / len(loader.frames))
    solver = DisplacementRecursiveSolver(
        model, material, loader, bounds, device,
        spatial_ratio=1.0,
        temporal_ratio=temporal_ratio,
        seed=args.seed,
        component_weights=component_weights,
        lambda_data=1.0,
        lambda_equilibrium=lambda_equilibrium,
        lambda_bc_left=1.0,
        lambda_bc_right=1.0,
    )
    output_dir = (Path(args.output_dir) if args.output_dir else
                  script_dir / 'outputs_displacement_recursive' / args.model / f'seed_{args.seed}')
    solver.train(
        epochs=args.epochs,
        node_batch=args.node_batch,
        pde_time_samples=args.pde_times,
        learning_rate=1.0e-4,
        output_dir=output_dir,
    )


if __name__ == '__main__':
    main()

